In [1]:
import numpy as np
import pandas as pd


In [2]:
df=pd.read_csv('final_balanced_dataset2.csv')

In [3]:
df.head()

,Destination.Port,Protocol,Flow.Duration,Total.Fwd.Packets,Total.Backward.Packets,Total.Length.of.Fwd.Packets,Total.Length.of.Bwd.Packets,Average.Packet.Size,SYN.Flag.Count,ACK.Flag.Count,PSH.Flag.Count,ProtocolName
0,443,6,81841.0,8.0,5.0,332.0,3648.0,306.153846,0,0,1,SSL
1,443,6,93.0,1.0,1.0,0.0,0.0,0.000000,0,1,0,SSL
2,443,6,11715025.0,358.0,418.0,22040.0,365142.0,499.154600,1,1,0,SSL
3,443,6,84902.0,1.0,1.0,0.0,0.0,0.000000,0,1,0,SSL
4,443,6,11353359.0,19.0,13.0,832.0,6504.0,229.250000,0,0,1,SSL


In [4]:
df['ProtocolName'].value_counts()

ProtocolName
SSL         2000
HTTP        2000
SSH         2000
GMAIL       2000
IP_ICMP     2000
DNS         2000
FTP_DATA    2000
Name: count, dtype: int64

In [5]:
df.describe()

,Destination.Port,Protocol,Flow.Duration,Total.Fwd.Packets,Total.Backward.Packets,Total.Length.of.Fwd.Packets,Total.Length.of.Bwd.Packets,Average.Packet.Size,SYN.Flag.Count,ACK.Flag.Count,PSH.Flag.Count
count,14000.000000,14000.000000,1.400000e+04,14000.000000,14000.000000,1.400000e+04,1.400000e+04,14000.000000,14000.000000,14000.000000,14000.000000
mean,5913.247643,6.618857,3.020444e+07,248.689567,200.502223,2.403257e+05,4.623357e+04,195.803425,0.232357,0.561143,0.297643
std,16117.200456,4.518570,4.479672e+07,937.079843,1005.845589,2.076032e+06,9.349372e+05,338.934435,0.422351,0.496265,0.457238
min,0.000000,0.000000,7.121188e-01,0.700148,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,20.000000,6.000000,3.870000e+02,2.165764,1.203753,5.838525e+00,0.000000e+00,6.000000,0.000000,0.000000,0.000000
50%,53.000000,6.000000,5.319065e+05,5.298151,5.000000,6.376753e+02,1.770000e+02,73.911481,0.000000,1.000000,0.000000
75%,443.000000,6.000000,5.470667e+07,20.000000,20.000000,3.527250e+03,4.937000e+03,219.498854,0.000000,1.000000,1.000000
max,65521.000000,17.000000,1.497217e+08,47427.000000,64191.000000,1.277374e+08,1.046456e+08,3733.000000,1.000000,1.000000,1.000000


In [6]:
df.isna().sum()

Destination.Port               0
Protocol                       0
Flow.Duration                  0
Total.Fwd.Packets              0
Total.Backward.Packets         0
Total.Length.of.Fwd.Packets    0
Total.Length.of.Bwd.Packets    0
Average.Packet.Size            0
SYN.Flag.Count                 0
ACK.Flag.Count                 0
PSH.Flag.Count                 0
ProtocolName                   0
dtype: int64

In [7]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

df['ProtocolLabel']=le.fit_transform(df['ProtocolName'])

print(dict(zip(
            le.classes_,
            le.transform(le.classes_)
            )))

{'DNS': np.int64(0), 'FTP_DATA': np.int64(1), 'GMAIL': np.int64(2), 'HTTP': np.int64(3), 'IP_ICMP': np.int64(4), 'SSH': np.int64(5), 'SSL': np.int64(6)}


In [8]:
y = df["ProtocolLabel"]

X = df.drop(
    columns=[
        "ProtocolName",
        "ProtocolLabel"
    ]
)



In [9]:
y

0        6
1        6
2        6
3        6
4        6
        ..
13995    1
13996    1
13997    1
13998    1
13999    1
Name: ProtocolLabel, Length: 14000, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)

X_test_scaled=scaler.transform(X_test)

In [12]:
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

models = {
    "RandomForest": RandomForestClassifier(
        random_state=42
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [13]:
param_grids = {

    "RandomForest": {

        "model__n_estimators":
            [100, 200, 300, 500],

        "model__max_depth":
            [5, 10, 20, None],

        "model__min_samples_split":
            [2, 5, 10],

        "model__min_samples_leaf":
            [1, 2, 4],

        "model__max_features":
            ["sqrt", "log2"]
    },

    "GradientBoosting": {

        "model__n_estimators":
            [100, 200, 300],

        "model__learning_rate":
            [0.01, 0.05, 0.1, 0.2],

        "model__max_depth":
            [3, 5, 7],

        "model__subsample":
            [0.8, 0.9, 1.0]
    }
}

In [20]:
from sklearn.pipeline import Pipeline

best_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    random_search = RandomizedSearchCV(
        estimator=pipeline,

        param_distributions=
        param_grids[name],

        n_iter=5,

        cv=5,

        scoring="accuracy",

        verbose=2,

        random_state=42,

        n_jobs=-1
    )

    random_search.fit(
        X_train,
        y_train
    )

    best_models[name] = (
        random_search.best_estimator_
    )

    print(
        f"\nBest Params for {name}:"
    )

    print(
        random_search.best_params_
    )

    print(
        f"Best CV Score: "
        f"{random_search.best_score_:.4f}"
    )


Training RandomForest...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Best Params for RandomForest:
{'model__n_estimators': 100, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': None}
Best CV Score: 0.9398

Training GradientBoosting...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Best Params for GradientBoosting:
{'model__subsample': 1.0, 'model__n_estimators': 200, 'model__max_depth': 7, 'model__learning_rate': 0.1}
Best CV Score: 0.9425


In [22]:
from sklearn.metrics import accuracy_score,classification_report
for name, model in best_models.items():

    y_pred = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    print("\n" + "="*50)
    print(name)
    print("="*50)

    print(
        "Accuracy:",
        accuracy
    )

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=
            le.classes_
        )
    )


RandomForest
Accuracy: 0.9282142857142858
              precision    recall  f1-score   support

         DNS       1.00      1.00      1.00       400
    FTP_DATA       0.98      0.91      0.94       400
       GMAIL       0.84      0.81      0.83       400
        HTTP       0.87      0.92      0.89       400
     IP_ICMP       1.00      1.00      1.00       400
         SSH       1.00      1.00      1.00       400
         SSL       0.82      0.87      0.84       400

    accuracy                           0.93      2800
   macro avg       0.93      0.93      0.93      2800
weighted avg       0.93      0.93      0.93      2800


GradientBoosting
Accuracy: 0.9282142857142858
              precision    recall  f1-score   support

         DNS       1.00      1.00      1.00       400
    FTP_DATA       0.98      0.92      0.95       400
       GMAIL       0.84      0.81      0.82       400
        HTTP       0.89      0.91      0.90       400
     IP_ICMP       1.00      1.00      1.0

In [24]:
import joblib

best_model_name = max(
    best_models,
    key=lambda x:
    best_models[x].score(
        X_test,
        y_test
    )
)

best_model = best_models[
    best_model_name
]

print(
    "Best Model:",
    best_model_name
)

joblib.dump(
    best_model,
    "best_protocol_model.pkl"
)

joblib.dump(
    le,
    "label_encoder.pkl"
)

Best Model: RandomForest


['label_encoder.pkl']

In [26]:
rf_model = best_models["RandomForest"].named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(10))

                       Feature  Importance
0             Destination.Port    0.399042
1                     Protocol    0.117801
2                Flow.Duration    0.085429
7          Average.Packet.Size    0.079833
3            Total.Fwd.Packets    0.063397
8               SYN.Flag.Count    0.058510
5  Total.Length.of.Fwd.Packets    0.057998
4       Total.Backward.Packets    0.049476
6  Total.Length.of.Bwd.Packets    0.038487
9               ACK.Flag.Count    0.034645
